# CSA batch-level feature sharing — Accelerate / FP32 / T4×2

InternetON, GPU T4×2, Save & Run All. Attach the SAME four Datasets: visual-cache, vuthetam/mscoco-2014, csa-semantic-banks-v130, csa-prompt-full-karpathy-v175. No Secret/checkpoint/newcache. Lastbudget6hown/30hNgân. Parent75min stopcap includes setup, not ETA/billing.

Same restored2OSA+1ORA/dual-memory3layerdecoder/promptadapters. Unique images encode ONCE per GPU local batch, then both memories index_select back to caption order. Gradients sum across all captions; decoder still processes all32globalcaptions. No parameters added, no FP16. This supersedes the previous native-DDP benchmark.

Accelerate1.15.0 manages devices/DDP/model+optimizer preparation/backward/clipping. FP32/no mixed precision, accumulation1. Torchrun launches two processes only. NumPy event batches are already rank-sliced: no Accelerate DataLoader sharding. Explicit free_memory between fresh passes prevents retaining previous models. Wheel pinned bySHA, isolated --no-deps install; never reinstallTorch/HFhub. Runtime version/DDP/precision recorded, import checked before staging. No speedup promise versus nativeDDP.

Two TRAIN256/1280event orders: original random captions and image-grouped. Each order has ordinary/shared/shared/ordinary passes,40updates/pass,10warmup+30timed;320updates total with fresh identical initialization every pass. Matched event IDs WITHIN each order; all1280events consumed per pass. Random batches may have few repeated images and sharing may be slower. Record actual encoded-image count, synchronized slowest-rank latency and peak memory.

Numerical guards BEFORE timing: dropoutOFF, ordinary/shared logits and DDP token loss/all parameter gradients versus full-global ordinary reference; include unequal16/11tail27. Encoder row hook verifies sharing actually occurs. Training speed uses dropout.2 ON: shared encoder dropout becomes correlated across captions of an image. Training trajectories are not claimed identical. Grouped sampling changes caption order and must be chosen explicitly for future full training.

V2 stopped at the first ORDINARY gradient guard, before sharing/timing. V3 keeps model/precision/optimizer/samplers/timing unchanged, uses ordinary rank-shaped microbatch gradient reference (same global token denominator) and scale-aware per-tensor diagnostics. Full-global ordinary scalar loss still checked. RelativeL2 limit.005 is stricter than V2.01; maxabsolute limit scales with reference magnitude plus3e-7. Detailed diagnostics saved even if rejected; no assumed GPU PASS. Do not label V2 a performance result.

No VALID/TEST/CIDEr/SCST/rerank/fulltraining/checkpoint. Author only NumPy/AST, actual Accelerate/GPU parity/performance pending. Send csa_feature_sharing_accelerate_v3_review.zip. No auto restart, cap extension or Torch reinstall.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
REVISION='fa9abcc2b1cc862c44dc7ad1233fea1b8247e197'
MAPPING={'experiments/clip_patch_baseline/csa_patch_model.py': 'csa_patch_model.py', 'experiments/clip_patch_baseline/csa_structure_data.py': 'csa_structure_data.py', 'experiments/clip_patch_baseline/csa_structure_runtime.py': 'csa_structure_runtime.py', 'experiments/clip_patch_baseline/run_csa_structure_smoke.py': 'run_csa_structure_smoke.py', 'experiments/clip_patch_baseline/test_csa_structure_data.py': 'test_csa_structure_data.py', 'experiments/clip_patch_baseline/semantic_data.py': 'semantic_data.py', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'semantic_patch_model.py', 'experiments/clip_patch_baseline/memory_fusion.py': 'memory_fusion.py', 'experiments/clip_patch_baseline/patch_data.py': 'patch_data.py', 'experiments/clip_patch_baseline/patch_model.py': 'patch_model.py', 'experiments/clip_patch_baseline/patch_beam.py': 'patch_beam.py', 'experiments/clip_patch_baseline/full_data.py': 'full_data.py', 'experiments/clip_patch_baseline/test_semantic_data.py': 'test_semantic_data.py', 'experiments/clip_patch_baseline/CSA_STRUCTURE_PROTOCOL.md': 'CSA_STRUCTURE_PROTOCOL.md', 'experiments/clip_patch_baseline/make_csa_structure_notebook.py': 'make_csa_structure_notebook.py', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/cuda_runtime_guard.py': 'cuda_runtime_guard.py', 'experiments/csa_paper/SAVE_BANK_V130_PACKAGE.json': 'SAVE_BANK_V130_PACKAGE.json', 'experiments/csa_paper/FULL_PROMPT_FRESH_V175_RESULT.json': 'FULL_PROMPT_FRESH_V175_RESULT.json', 'experiments/csa_paper/model.py': 'bank_layers.py', 'experiments/csa_paper/lemma_correction_review_text/bank_words_corrected.json': 'bank_words_corrected.json', 'experiments/clip_patch_baseline/csa_sharing_data.py': 'csa_sharing_data.py', 'experiments/clip_patch_baseline/csa_shared_model.py': 'csa_shared_model.py', 'experiments/clip_patch_baseline/run_csa_sharing_benchmark.py': 'run_csa_sharing_benchmark.py', 'experiments/clip_patch_baseline/test_csa_sharing_data.py': 'test_csa_sharing_data.py', 'experiments/clip_patch_baseline/test_csa_accelerate_contract.py': 'test_csa_accelerate_contract.py', 'experiments/clip_patch_baseline/gradient_agreement.py': 'gradient_agreement.py', 'experiments/clip_patch_baseline/test_gradient_agreement.py': 'test_gradient_agreement.py', 'experiments/clip_patch_baseline/CSA_SHARING_PROTOCOL.md': 'CSA_SHARING_PROTOCOL.md', 'experiments/clip_patch_baseline/make_csa_sharing_notebook.py': 'make_csa_sharing_notebook.py'}
HASHES={'experiments/clip_patch_baseline/csa_patch_model.py': '99b6441c4205b142256192cc3d840a905e00ecc1e730f30e1f5eb2c3cf8dc220', 'experiments/clip_patch_baseline/csa_structure_data.py': '3bce011b00f67d272444cdd4d6e17acbbb744c8d9614549fb06c73be79aba1a2', 'experiments/clip_patch_baseline/csa_structure_runtime.py': '85a9806799ad2d2cb5da0bb458c373af830881f5cd17770a597d4b0274309cd9', 'experiments/clip_patch_baseline/run_csa_structure_smoke.py': 'e4c5c9f7fa9ed1fb244b9745170d02f680df36a7305116c645fc66cc5530ab2c', 'experiments/clip_patch_baseline/test_csa_structure_data.py': '55b45cf9cb6570a8f3d8cff1bbca7b80a0a29fd8dd3d8a24bbf942abc5f7cf16', 'experiments/clip_patch_baseline/semantic_data.py': 'a57af931f9e6be85f6efeee668b09fcca14c8a3bd54ebd695d9014ede2ab0540', 'experiments/clip_patch_baseline/semantic_patch_model.py': '10a385859ea1389a71f14a7a76e16a6989feb73fd581ca89659eab50ec53a0f3', 'experiments/clip_patch_baseline/memory_fusion.py': 'a70947e22263ddd27a7896a94cfb35cfff73102370288c53f86529964ea8f087', 'experiments/clip_patch_baseline/patch_data.py': '3706ffec9c128c8fe12c4f21ba071939268fdd526ed614034cca8dfec55674d5', 'experiments/clip_patch_baseline/patch_model.py': 'dfe6b1d83bfc1e3c7961268fb104bc1f4092129e7d9400841568a3204dc57a70', 'experiments/clip_patch_baseline/patch_beam.py': '53fcb5b22bb9f05d795fbd4f539cfd4ff0af39c1dea679f7df1a933edf2754be', 'experiments/clip_patch_baseline/full_data.py': 'ab69cf61d4ad0456d9a869929750ab10f3a2a33b6c76683689539e57e4015a74', 'experiments/clip_patch_baseline/test_semantic_data.py': 'e9d9132db2e83b47e7f29cd1c465389cb4f16db2cd62a1cf27f048a06e29a572', 'experiments/clip_patch_baseline/CSA_STRUCTURE_PROTOCOL.md': '0519a80e74f7a3c4ed6a39978fe7cc6b864fec142fcb977520def08791328177', 'experiments/clip_patch_baseline/make_csa_structure_notebook.py': '68526f40aca7f4ba17691227db0ebec6ac0d9c12efe9c810cd820f245f44e777', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/cuda_runtime_guard.py': '640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773', 'experiments/csa_paper/SAVE_BANK_V130_PACKAGE.json': '826ba36628d6587860e2d548e950ea953bd4bbb8c0a8dc64393089dbc1e058e1', 'experiments/csa_paper/FULL_PROMPT_FRESH_V175_RESULT.json': '452e94530b07b117be6873f004404b558e29e87cb09e97e27a1548d0bfd5f81a', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/lemma_correction_review_text/bank_words_corrected.json': 'cec76a0229f30361235a4b64e9712ac6bb2e79fc62a9241079a7af3fa95d13ac', 'experiments/clip_patch_baseline/csa_sharing_data.py': '4d89428b8f833e34522bd51ec25640015edbebae4f0b5b81450058e2ebcb1243', 'experiments/clip_patch_baseline/csa_shared_model.py': '9599759a16e8570fbb625c88c821f2cf26776be8a317db650adcd34ef563cbf8', 'experiments/clip_patch_baseline/run_csa_sharing_benchmark.py': '3e8da0d2056de0cf1b536341ce3a4933a8d2795977620a216dffd18b379ab05c', 'experiments/clip_patch_baseline/test_csa_sharing_data.py': '4fbda314a2d18677675793d578ac46fc0e3e59c5e10de19a32d29503a618f666', 'experiments/clip_patch_baseline/test_csa_accelerate_contract.py': 'f222141e8a67a8051bcd463f8df0ce2336335e44b16c8f254858a3cf9d3c6a77', 'experiments/clip_patch_baseline/gradient_agreement.py': 'dc06ffd922fccd9e14e860d07644dddcf320f7d9ac6410e3166cc4f53200971c', 'experiments/clip_patch_baseline/test_gradient_agreement.py': '2b2aa46091a5a752b3eab573f701c0f1e2849349d829e1c4d6a4d99ae73285a6', 'experiments/clip_patch_baseline/CSA_SHARING_PROTOCOL.md': '2bf4ab1f6e5b4dc92069198135b3016ee6cfec065bf6d8057fb2afc6e7caf463', 'experiments/clip_patch_baseline/make_csa_sharing_notebook.py': 'f19f2b0ee8f0a64838e6666ea050ebabca45ff6195cc8ddca4e8c1df45fc378c'}
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
BANK_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-semantic-banks-v130')
PROMPT_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-prompt-full-karpathy-v175')
OUTPUT=Path('/kaggle/working/csa_feature_sharing_accelerate_v3')
for p in (VISUAL_CACHE,COCO_JSON,BANK_ROOT,PROMPT_ROOT):
    if not p.exists():raise FileNotFoundError('Attach/correct all four inputs BEFORE GPU: '+str(p))
if OUTPUT.exists():raise RuntimeError('Preserve prior output; use fresh session/output path')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<2*1024**3:raise RuntimeError('Need2GiB for small TRAIN stage/logs')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='csa-structure-',dir='/tmp'));LOG=OUTPUT/'notebook.log'
PARENT_DEADLINE=time.monotonic()+4500 #75min STOPcap including setup; not ETA/billing.
os.environ['PYTHONPATH']=str(SOURCE);os.environ['OMP_NUM_THREADS']='1'
CONFIG=SOURCE/'config.json'
cfg=dict(output=str(OUTPUT),stage=str(SOURCE/'stage'),dataset_json=str(COCO_JSON),visual_cache=str(VISUAL_CACHE),
         bank_root=str(BANK_ROOT),prompt_root=str(PROMPT_ROOT),updates=40,warmup=10,prepare_seconds=2100,worker_seconds=2100)
def write_config():
    CONFIG.write_text(json.dumps(cfg,indent=2));(OUTPUT/'protocol.json').write_text(json.dumps(cfg,indent=2))
write_config()

ACCEL_DEPS=SOURCE/'accelerate-deps'
os.environ['PYTHONPATH']=str(ACCEL_DEPS)+os.pathsep+str(SOURCE)


In [ ]:
def run(args, timeout=1800):
    if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError('Parent cap exceeded')
    timeout=min(timeout,PARENT_DEADLINE-time.monotonic())
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('CSA structure active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Smoke wall cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    for name,destination in MAPPING.items():
        remaining=PARENT_DEADLINE-time.monotonic()
        if remaining<=0:raise TimeoutError('Source download parent cap')
        raw=urllib.request.urlopen('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/'+name,timeout=min(90,remaining)).read()
        if hashlib.sha256(raw).hexdigest()!=HASHES[name]:raise ValueError('Changed source: '+name)
        (SOURCE/destination).write_bytes(raw)
    (OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=REVISION,sha256=HASHES),indent=2))
    run([sys.executable,'-u',SOURCE/'test_csa_sharing_data.py'],120)
    run([sys.executable,'-u',SOURCE/'test_gradient_agreement.py'],120)
    run([sys.executable,'-u',SOURCE/'cuda_runtime_guard.py','--output',OUTPUT],60)

    run([sys.executable,'-u',SOURCE/'test_csa_accelerate_contract.py'],120)
    run([sys.executable,'-m','pip','download','-q','--no-deps','--only-binary=:all:','--dest',SOURCE,'accelerate==1.15.0'],180)
    wheel=SOURCE/'accelerate-1.15.0-py3-none-any.whl'
    if hashlib.sha256(wheel.read_bytes()).hexdigest()!='97eacca0b73e45cb867dbf8c5d5d4dc32219544300e0c8992c7334dc2ef33cec':raise ValueError('Changed Accelerate wheel')
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',ACCEL_DEPS,wheel],180)
    probe="import accelerate;from pathlib import Path;assert accelerate.__version__=='1.15.0';assert Path(accelerate.__file__).resolve().is_relative_to(Path("+repr(str(ACCEL_DEPS))+"));print('Pinned isolatedAccelerate',accelerate.__version__,flush=True)"
    run([sys.executable,'-c',probe],120)
    (OUTPUT/'accelerate_install.json').write_text(json.dumps(dict(version='1.15.0',wheel_sha256=hashlib.sha256(wheel.read_bytes()).hexdigest(),isolated_target=str(ACCEL_DEPS),dependencies_reinstalled=False)))

    run([sys.executable,'-u',SOURCE/'run_csa_sharing_benchmark.py','prepare','--config',CONFIG],2100)
    cfg['worker_seconds']=min(2100,PARENT_DEADLINE-time.monotonic()-120)
    if cfg['worker_seconds']<300:raise TimeoutError('Insufficient remaining parent budget; preserve log/stage')
    write_config()
    run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2',
         '--max_restarts=0',SOURCE/'run_csa_sharing_benchmark.py','worker','--config',CONFIG],cfg['worker_seconds']+30)
    complete=json.loads((OUTPUT/'result.json').read_text())['status']=='CSA_FEATURE_SHARING_ACCELERATE_BENCHMARK_COMPLETE'
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc());raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,full_training_complete=False,test_used=False,validation_used=False,SCST=False,reranking=False)))
    review=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for p in OUTPUT.rglob('*'):
            if p.is_file() and p.suffix in ('.json','.jsonl','.txt','.log'):z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',review,flush=True)
    print('ACCELERATE SHARING BENCHMARK COMPLETE; no full training/CIDEr result.' if complete else 'INCOMPLETE; preserve log/output, no automatic retry.',flush=True)
